In [1]:
import csv
import json
import re
from pathlib import Path
from collections import OrderedDict, defaultdict
from typing import Dict, List, Set, Tuple

# -----------------------------
# INPUTS
# -----------------------------
CSV_PATH = Path(r"app_total_columns.csv")
JSONL_PATH = Path(r"..\normalized_PII_results\GPT-5.1\app_level\app_level.jsonl")
OUT_TEX = Path("RQ2_search_space_reduction.tex")

# Locked app order + labels (table ID and display name)
APP_NAME_PLAIN = OrderedDict([
    ("A1", "WhatsApp"),
    ("A2", "Snapchat"),
    ("A3", "Telegram"),
    ("A4", "Google Maps"),
    ("A5", "Samsung Internet"),
    ("I1", "WhatsApp"),
    ("I2", "Contacts"),
    ("I3", "Apple Messages"),
    ("I4", "Safari"),
    ("I5", "Calendar"),
])

APP_CODE_RE = re.compile(r"^(A|I)\d+$", re.IGNORECASE)

# -----------------------------
# CORE RULE (CONFIRMED BY YOU)
# -----------------------------
# Cols Scanned (Extraction) = unique union of all source_columns across all PII types for that app.


def get_app_code_from_db_path(db_path: str) -> str:
    """
    Extract app code from db_path.
    Supports:
      - "selectedDBs\\A1" -> A1
      - "selectedDBs\\A1_msgstore.db" -> A1
      - "A1_something" -> A1
      - "A1-something" -> A1
    Hard-fails if app code cannot be derived confidently.
    """
    p = Path(db_path)

    # Prefer last path component (folder or filename stem)
    last = p.name
    stem = Path(last).stem

    # If last component itself is a code (e.g., folder named A1)
    if APP_CODE_RE.match(last.strip()):
        return last.strip().upper()
    if APP_CODE_RE.match(stem.strip()):
        return stem.strip().upper()

    # Split stem on separators
    for sep in ("_", "-"):
        if sep in stem:
            candidate = stem.split(sep, 1)[0].strip()
            if APP_CODE_RE.match(candidate):
                return candidate.upper()

    raise ValueError(f"Cannot derive app_code from db_path={db_path!r} (last={last!r}, stem={stem!r})")


def read_candidate_totals(csv_path: Path) -> Dict[str, int]:
    """
    Reads candidate totals. Hard-fails on missing required columns or bad ints.
    Expects headers: app_code, total_columns (app_name is allowed but unused here).
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    totals: Dict[str, int] = {}
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError("CSV has no header row")

        required = {"app_code", "total_columns"}
        missing = required - set(h.strip() for h in reader.fieldnames if h)
        if missing:
            raise ValueError(f"CSV missing required columns: {sorted(missing)}. Found: {reader.fieldnames}")

        for row_no, row in enumerate(reader, start=2):
            code = (row.get("app_code") or "").strip().upper()
            tc = (row.get("total_columns") or "").strip()
            if not code:
                raise ValueError(f"CSV row {row_no}: empty app_code")
            if not APP_CODE_RE.match(code):
                raise ValueError(f"CSV row {row_no}: invalid app_code={code!r}")
            if not tc:
                raise ValueError(f"CSV row {row_no}: empty total_columns for app_code={code}")
            try:
                total = int(tc)
            except ValueError as e:
                raise ValueError(f"CSV row {row_no}: total_columns not int for app_code={code}: {tc!r}") from e
            if total < 0:
                raise ValueError(f"CSV row {row_no}: total_columns negative for app_code={code}: {total}")
            if code in totals:
                raise ValueError(f"CSV row {row_no}: duplicate app_code={code}")
            totals[code] = total

    return totals


def read_scanned_cols_from_app_jsonl(jsonl_path: Path) -> Tuple[Dict[str, int], Dict[str, int], Dict[str, Set[str]]]:
    """
    Reads JSONL and computes:
      - scanned_counts: app_code -> count(unique union of source_columns across all PII types)
      - record_counts: app_code -> number of JSONL records observed for that app
      - scanned_sets:  app_code -> set of unique source_columns (for validation/inspection)

    Hard-fails on:
      - malformed JSON
      - missing/invalid db_path
      - source_columns not list (if present)
      - source_columns elements not strings
      - cannot derive app_code
    """
    if not jsonl_path.exists():
        raise FileNotFoundError(f"JSONL not found: {jsonl_path}")

    record_counts: Dict[str, int] = defaultdict(int)
    scanned_sets: Dict[str, Set[str]] = defaultdict(set)

    with jsonl_path.open("r", encoding="utf-8") as f:
        for line_no, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON in {jsonl_path} line {line_no}: {e}") from e

            if not isinstance(rec, dict):
                raise ValueError(f"JSONL line {line_no}: expected object/dict, got {type(rec).__name__}")

            db_path = rec.get("db_path", None)
            if not isinstance(db_path, str) or not db_path.strip():
                raise ValueError(f"JSONL line {line_no}: missing/invalid db_path")

            app = get_app_code_from_db_path(db_path)

            record_counts[app] += 1

            cols = rec.get("source_columns", [])
            if cols is None:
                cols = []
            if not isinstance(cols, list):
                raise ValueError(f"JSONL line {line_no}: source_columns must be a list, got {type(cols).__name__}")

            for c in cols:
                if not isinstance(c, str):
                    raise ValueError(f"JSONL line {line_no}: source_columns contains non-string: {c!r}")
                s = c.strip()
                if s:
                    scanned_sets[app].add(s)

    scanned_counts = {app: len(s) for app, s in scanned_sets.items()}
    # Ensure apps with records but no columns show up with 0
    for app in record_counts:
        scanned_counts.setdefault(app, 0)
        scanned_sets.setdefault(app, set())

    return scanned_counts, dict(record_counts), dict(scanned_sets)


def format_reduction(total: int, scanned: int) -> float:
    """
    Reduction (%) = (1 - scanned/total) * 100
    Hard-fails if total <= 0 or scanned < 0 or scanned > total (integrity).
    """
    if total <= 0:
        raise ValueError(f"Invalid total_columns={total}; must be > 0 for reduction computation")
    if scanned < 0:
        raise ValueError(f"Invalid scanned={scanned}; must be >= 0")
    if scanned > total:
        raise ValueError(f"Integrity error: scanned ({scanned}) > total_columns ({total})")
    return (1.0 - (scanned / total)) * 100.0


def latex_escape(s: str) -> str:
    """
    Minimal LaTeX escaping for safety.
    """
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    out = []
    for ch in s:
        out.append(repl.get(ch, ch))
    return "".join(out)


def build_latex_table(candidate_totals: Dict[str, int], scanned_counts: Dict[str, int]) -> str:
    lines: List[str] = []
    lines.append(r"\begin{table}[th]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Reduction of effective extraction space via hypothesis-driven planning."
        r" Total candidate columns are counted over all tables in the selected databases"
        r" for each application, while reductions correspond to columns exhaustively scanned"
        r" during row-level PII extraction.}"
    )
    lines.append(r"\label{tab:search_space_reduction}")
    lines.append(r"\small")
    lines.append(r"\begin{tabular}{|l|l|p{1.3cm}|p{1.7cm}|p{1.0cm}|}")
    lines.append(r"\hline")
    lines.append(
        r"\textbf{ID} & \textbf{Apps} & \textbf{Candidate Cols (Total)} & "
        r"\textbf{Cols Scanned (Extraction)} & \textbf{Reduc. (\%)} \\"
    )
    lines.append(r"\hline")

    # Deterministic order: use APP_NAME_PLAIN, then any extras (should not happen, but handled)
    app_order = list(APP_NAME_PLAIN.keys())
    extras = sorted(set(candidate_totals) - set(app_order))
    app_order += extras

    for app in app_order:
        if app not in candidate_totals:
            continue

        app_name = APP_NAME_PLAIN.get(app, app)
        total = candidate_totals[app]
        scanned = int(scanned_counts.get(app, 0))

        reduc = format_reduction(total, scanned)

        lines.append(
            f"{latex_escape(app)} & {latex_escape(app_name)} & {total} & {scanned} & {reduc:.2f}\\% \\\\"
        )
        lines.append(r"\hline")

    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


def build_plaintext_table(candidate_totals: Dict[str, int], scanned_counts: Dict[str, int]) -> str:
    headers = ["ID", "Apps", "CandidateCols", "ColsScanned", "Reduc(%)"]

    app_order = list(APP_NAME_PLAIN.keys())
    extras = sorted(set(candidate_totals) - set(app_order))
    app_order += extras

    rows = [headers]
    for app in app_order:
        if app not in candidate_totals:
            continue
        app_name = APP_NAME_PLAIN.get(app, app)
        total = candidate_totals[app]
        scanned = int(scanned_counts.get(app, 0))
        reduc = format_reduction(total, scanned)
        rows.append([app, app_name, str(total), str(scanned), f"{reduc:.2f}%"])

    widths = [0] * len(headers)
    for r in rows:
        for i, cell in enumerate(r):
            widths[i] = max(widths[i], len(cell))

    def fmt_row(r):
        parts = []
        for i, cell in enumerate(r):
            if i in (0, 1):
                parts.append(cell.ljust(widths[i]))
            else:
                parts.append(cell.rjust(widths[i]))
        return "  ".join(parts)

    out = [fmt_row(rows[0]), "-" * len(fmt_row(rows[0]))]
    for r in rows[1:]:
        out.append(fmt_row(r))
    return "\n".join(out)


def build_validation_report(
    candidate_totals: Dict[str, int],
    scanned_counts: Dict[str, int],
    record_counts: Dict[str, int],
) -> str:
    """
    Prints per-app sanity stats.
    Hard-fails already happen in format_reduction (scanned > total).
    """
    lines: List[str] = []
    lines.append("Validation report:")
    lines.append("  Rule: ColsScanned = unique union of source_columns across all PII types for the app")
    lines.append("")

    app_order = list(APP_NAME_PLAIN.keys())
    extras = sorted(set(candidate_totals) - set(app_order))
    app_order += extras

    for app in app_order:
        if app not in candidate_totals:
            continue
        total = candidate_totals[app]
        scanned = int(scanned_counts.get(app, 0))
        recs = int(record_counts.get(app, 0))
        reduc = format_reduction(total, scanned)
        lines.append(f"  {app}: records={recs}, total_cols={total}, scanned_unique_cols={scanned}, reduction={reduc:.2f}%")

    # Also warn if JSONL contains apps not in CSV (not fatal, but surfaced)
    extra_jsonl_apps = sorted(set(record_counts) - set(candidate_totals))
    if extra_jsonl_apps:
        lines.append("")
        lines.append("  WARNING: JSONL contains app_codes not present in CSV:")
        for a in extra_jsonl_apps:
            lines.append(f"    - {a} (records={record_counts.get(a,0)}, scanned={scanned_counts.get(a,0)})")

    return "\n".join(lines)


if __name__ == "__main__":
    candidate_totals = read_candidate_totals(CSV_PATH)
    scanned_counts, record_counts, _scanned_sets = read_scanned_cols_from_app_jsonl(JSONL_PATH)

    # Ensure every CSV app code is present in scanned_counts (0 if none)
    for app in candidate_totals:
        scanned_counts.setdefault(app, 0)
        record_counts.setdefault(app, 0)

    # Build outputs (integrity checks happen inside format_reduction)
    tex = build_latex_table(candidate_totals, scanned_counts)
    OUT_TEX.write_text(tex, encoding="utf-8")

    print(build_plaintext_table(candidate_totals, scanned_counts))
    print()
    print(build_validation_report(candidate_totals, scanned_counts, record_counts))
    print(f"\nWrote LaTeX: {OUT_TEX.resolve()}")


ID  Apps              CandidateCols  ColsScanned  Reduc(%)
----------------------------------------------------------
A1  WhatsApp                   1627           40    97.54%
A2  Snapchat                    842          161    80.88%
A3  Telegram                   1197            0   100.00%
A4  Google Maps                  71            2    97.18%
A5  Samsung Internet            173           43    75.14%
I1  WhatsApp                    328           44    86.59%
I2  Contacts                    219           17    92.24%
I3  Apple Messages              181           39    78.45%
I4  Safari                       72            0   100.00%
I5  Calendar                    539           35    93.51%

Validation report:
  Rule: ColsScanned = unique union of source_columns across all PII types for the app

  A1: records=5, total_cols=1627, scanned_unique_cols=40, reduction=97.54%
  A2: records=5, total_cols=842, scanned_unique_cols=161, reduction=80.88%
  A3: records=5, total_cols=1197, s